# 🍏 Health Resource Search Agent Tutorial 🍎

Welcome to the **Health Resource Search Agent** tutorial! We'll use **Microsoft Foundry** SDKs to build an assistant that can:

1. **Upload** health and recipe files into a vector store.
2. **Create an Agent** with a **File Search** tool.
3. **Search** these documents for relevant dietary info.
4. **Answer** health and wellness questions (with disclaimers!).

### ⚠️ Important Medical Disclaimer ⚠️
> **All health information in this notebook is for general educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of a qualified healthcare professional with any questions you may have.

## Prerequisites
- Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
- **Roles**  
  1. **Foundry User** on your Microsoft Foundry project.
  2. If your project uses customer-managed storage or search, assign the data-plane roles required by those connected resources.

## Let's Get Searching!
We'll show you how to upload some sample files, create a vector store for them, then spin up an agent that can search these resources for dietary guidelines, recipes, and more. Enjoy!

<img src="./seq-diagrams/3-file-search.png" width="30%"/>


## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
Here we import needed libraries, load environment variables from `.env`, and initialize our **AIProjectClient**. Let's do this! 🎉

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find .env. Complete Lab 00 and place it in the repository root."
    )

load_dotenv(env_path)
tenant_id = os.environ.get("TENANT_ID")
ai_foundry_project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.environ.get("MODEL_DEPLOYMENT_NAME")
missing_variables = [
    name
    for name, value in {
        "TENANT_ID": tenant_id,
        "AI_FOUNDRY_PROJECT_ENDPOINT": ai_foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]
if missing_variables:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_variables)}")

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(
    endpoint=ai_foundry_project_endpoint,
    credential=credential,
)
openai_client = project_client.get_openai_client()
print(f"📁 Environment loaded from: {env_path}")
print("✅ Successfully initialized AIProjectClient and OpenAI client")

## 2. Prepare Sample Files 🍲🗒
We'll create some dummy .md files (for recipes and guidelines). Then we'll store them in a vector store for searching.


In [ ]:
def create_sample_files():
    recipes_md = """# Healthy Recipes Database

## Gluten-Free Recipes
1. Quinoa Bowl
   - Ingredients: quinoa, vegetables, olive oil
   - Instructions: Cook quinoa and add vegetables.
2. Rice Pasta with Vegetables
   - Ingredients: rice pasta and mixed vegetables
   - Instructions: Boil pasta and sauté vegetables.

## Diabetic-Friendly Recipes
1. Low-Carb Stir Fry
   - Ingredients: chicken, vegetables, tamari sauce
2. Greek Salad
   - Ingredients: cucumber, tomatoes, feta, olives

## Heart-Healthy Recipes
1. Baked Salmon
   - Ingredients: salmon, lemon, herbs
2. Mediterranean Bowl
   - Ingredients: chickpeas, vegetables, tahini
"""

    guidelines_md = """# Dietary Guidelines

## General Guidelines
- Eat a variety of foods.
- Control portion sizes.
- Stay hydrated.

## Special Diets
1. Gluten-Free Diet
   - Avoid wheat, barley, and rye.
2. Diabetic Diet
   - Monitor carbohydrate intake and choose lower-glycemic foods.
3. Heart-Healthy Diet
   - Limit saturated fats and choose lean proteins.
"""

    recipes_filename = os.environ.get("RECIPES_FILENAME", "recipes.md")
    guidelines_filename = os.environ.get("GUIDELINES_FILENAME", "guidelines.md")

    Path(recipes_filename).write_text(recipes_md, encoding="utf-8")
    Path(guidelines_filename).write_text(guidelines_md, encoding="utf-8")
    print(f"📄 Created sample files: {recipes_filename}, {guidelines_filename}")
    return [recipes_filename, guidelines_filename]


sample_files = create_sample_files()

#### ✨ Note on Permissions

Your account needs the **Foundry User** role on the project. If your project uses a standard agent environment with customer-managed storage or search, also assign the data-plane roles required by those connected resources.

## 3. Create a Vector Store 📚
We'll upload our newly created files and group them into a single vector store for searching. This is how the agent can later find relevant text.

In [ ]:
def create_vector_store(files, store_name="health_resources_example"):
    vector_store = openai_client.vector_stores.create(name=store_name)
    uploaded_ids = []

    for file_path in files:
        with open(file_path, "rb") as source_file:
            vector_file = openai_client.vector_stores.files.upload_and_poll(
                vector_store_id=vector_store.id,
                file=source_file,
            )
        uploaded_ids.append(vector_file.id)
        print(f"✅ Uploaded and indexed: {file_path} -> {vector_file.id}")

    print(f"🎉 Created vector store '{store_name}', ID: {vector_store.id}")
    return vector_store, uploaded_ids


vector_store, file_ids = create_vector_store(sample_files)

## 4. Create the Health Resource Agent 🔎
We use a **FileSearchTool** pointing to our newly created vector store, then create the Agent with instructions about disclaimers, dietary help, etc.

In [ ]:
def create_health_resource_agent(vector_store_id):
    file_search = FileSearchTool(vector_store_ids=[vector_store_id])
    agent = project_client.agents.create_version(
        agent_name="health-search-agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="""
            You are a health resource advisor with access to dietary and recipe files.
            Always state that you are not a doctor, cite the files you use, focus on
            general nutrition or recipe guidance, and encourage professional
            consultation for individual medical needs.
            """,
            tools=[file_search],
        ),
    )
    print(f"🎉 Created agent {agent.name}, version: {agent.version}")
    return agent


health_agent = create_health_resource_agent(vector_store.id)

## 5. Searching Health Resources 🏋️👩‍🍳

Create one Conversation and send several Responses requests. The `FileSearchTool` remains connected to the vector store for every turn.

In [ ]:
search_conversation = openai_client.conversations.create()
search_responses = []

queries = [
    "Could you suggest a gluten-free lunch recipe?",
    "Show me some heart-healthy meal ideas.",
    "What guidelines do you have for someone with diabetes?",
]

for query in queries:
    print(f"🔎 {query}")
    response = openai_client.responses.create(
        conversation=search_conversation.id,
        input=query,
        extra_body={
            "agent_reference": {
                "name": health_agent.name,
                "type": "agent_reference",
            }
        },
    )
    search_responses.append((query, response))
    print(f"🤖 {response.output_text}\n")

## 6. View Results & Citations 📄

File Search citations appear as `file_citation` annotations on output text blocks. Display the cited filename and file ID for source transparency.

In [ ]:
def display_response_citations(query, response):
    print(f"\nUSER: {query}")
    print(f"ASSISTANT: {response.output_text}")

    citations = []
    for item in response.output:
        if item.type != "message":
            continue
        for block in item.content:
            if block.type != "output_text":
                continue
            for annotation in block.annotations:
                if annotation.type == "file_citation":
                    citations.append((annotation.filename, annotation.file_id))

    if citations:
        print("📎 Citations:")
        for filename, file_id in citations:
            print(f"- {filename} ({file_id})")
    else:
        print("📎 No file citations were returned for this response.")


for query, response in search_responses:
    display_response_citations(query, response)

## 7. Cleanup & Best Practices 🧹
We'll optionally remove the vector store, the uploaded files, and the agent. In a production environment, you might keep them around longer. Meanwhile, here are some tips:

1. **Resource Management**
   - Keep files grouped by category, regularly prune old or irrelevant files.
   - Clear out test agents or vector stores once you're done.

2. **Search Queries**
   - Provide precise or multi-part queries.
   - Consider synonyms or alternative keywords ("gluten-free" vs "celiac").
   
3. **Health Information**
   - Always disclaim that you are not a medical professional.
   - Encourage users to see doctors for specific diagnoses.

4. **Performance**
   - Keep an eye on vector store size.
   - Evaluate search accuracy with `azure-ai-evaluation`!


In [ ]:
def cleanup_all():
    openai_client.conversations.delete(conversation_id=search_conversation.id)
    print("🗑️ Deleted Conversation.")

    project_client.agents.delete_version(
        agent_name=health_agent.name,
        agent_version=health_agent.version,
    )
    print("🗑️ Deleted health resource agent version.")

    openai_client.vector_stores.delete(vector_store.id)
    print("🗑️ Deleted vector store.")

    for file_id in file_ids:
        openai_client.files.delete(file_id)
    print("🗑️ Deleted uploaded files.")

    for sample_file in sample_files:
        if os.path.exists(sample_file):
            os.remove(sample_file)
    print("🗑️ Deleted local sample files.")

    openai_client.close()
    project_client.close()
    credential.close()
    print("✅ Cleanup completed!")


cleanup_all()

# Congratulations! 🎉

You created an OpenAI vector store, uploaded and indexed files with `vector_stores.files.upload_and_poll()`, connected a `FileSearchTool` to a versioned prompt agent, invoked it through Responses and Conversations, and inspected `file_citation` annotations. Delete vector stores, uploaded files, and test agent versions when the exercise is complete.